# S1 · Datos en la industria y sistemas de adquisición · Del sensor al archivo
**Sistemas de adquisición de datos · TSU en Electrónica y Pruebas · Jabil**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggt13/DAQ_TSU/blob/main/sesion_01_datos_y_adquisicion/S1_AC_CLASE_Colab_ALUMNO.ipynb)

## Objetivo
Seguir un dato por toda la **cadena de adquisición** —temperatura → resistencia del sensor → voltaje → cuenta del
convertidor → renglón de un archivo— y estimar cuánto información genera un sensor en un día.

## La situación
En la planta, un termistor NTC vigila la temperatura de un proceso. Está en un divisor de tensión con una resistencia
fija `RF` y lo lee un convertidor de 10 bits (como el de un Arduino); cada lectura se guarda como un renglón de texto.
Cada alumno tiene su propia resistencia fija, su temperatura de trabajo y su intervalo de registro.

## En qué consiste este cuaderno
| Parte | Tiempo | Qué haces | Qué produce |
|---|---|---|---|
| **A · Actividad en clase** | 30 min | Cuatro ejercicios (★ primero) sobre **tu** cadena: resistencia, voltaje, cuenta y volumen de datos. En cada uno escribes tu resultado, la sustitución con tus números y su unidad, y lo que predice | Tu cadena completa con números |
| **B · Práctica** | 80 min | Una semana de temperatura real de Guadalajara pasa por **tu** cadena; reconstruyes la temperatura desde las cuentas y mides cuánto se pierde | Gráficas firmadas, tu conclusión y el archivo `S1_<código>.csv` |
| **Hoja de resultados** | en la práctica | Importas el CSV en tu hoja de cálculo de Google y contestas las preguntas | El reporte que entregas |

## Qué entregas y cómo se califica
1. **Este cuaderno**, al terminar la sesión, con todo ejecutado. ★ 3 pts · ejercicios 2, 3 y 4, 2 pts cada uno ·
   cierre 1 pt. Se califica que el resultado sea correcto para **tus** datos y que la sustitución y la predicción
   sean tuyas.
2. **La hoja de resultados** con el reporte, en 7 días naturales.

## Reglas
- **Archivo → Guardar una copia en Drive** antes de empezar; trabaja en tu copia.
- Completa sólo lo marcado con ✎. Ninguna celda avanza si falta algo: el mensaje dice qué.
- La revisión automática dice si el ejercicio está **completo**, no si está **bien**; eso lo califica el profesor.
- Si usas un asistente de IA, dilo en tu reporte. Las sustituciones y las respuestas llevan **tus** números.

## Cómo se usa este cuaderno · no necesitas saber programar
Sólo cambias lo que está después del signo **=** en los renglones marcados con ✎.

| Para… | Haz esto |
|---|---|
| Ejecutar una celda | Clic en ▶ a su izquierda (o Ctrl + Enter). Ejecuta las celdas en orden, de arriba hacia abajo |
| Escribir un número | Después del **=**, con punto decimal y sin comas de miles: `u_A = 0.0316`, `MEMORIA = 32000`. Usa al menos 3 cifras significativas |
| Escribir un texto | Entre comillas: `SUST_1 = "70 / 2^10 = 0.0684 °C"`. No borres las comillas |
| Calcular | Con calculadora o con tu hoja de cálculo. Si sabes Python puedes escribir la operación en vez del número: `+ − * /`, potencia `**`, raíz `np.sqrt(x)`, eˣ `np.exp(x)` |
| Ver tus datos | Están impresos debajo de la celda «Tus datos»; usa esos números |
| Un mensaje en rojo | Lee la **última línea**: dice qué falta. Corrige y vuelve a ejecutar esa celda |
| Antes de entregar | **Entorno de ejecución → Reiniciar y ejecutar todo**, y revisa que todo salga ✔ |

In [ ]:
#@title Herramientas del curso (no modificar)
"""Herramientas comunes del curso «Sistemas de adquisición de datos» 2027.

Identificación del alumno, datos personales a partir de su código, firma de las gráficas,
huella de los resultados y CSV firmado para la hoja de resultados.

Este archivo es la fuente única: los cuadernos de cada sesión copian su contenido en la celda
«Herramientas del curso». Si se corrige aquí, se regeneran los cuadernos.
"""
import csv
import datetime as _dt
import hashlib
import zlib

import numpy as np

try:
    from zoneinfo import ZoneInfo
    _ZONA = ZoneInfo("America/Mexico_City")
except Exception:          # Python sin zoneinfo: se usa la hora del sistema
    _ZONA = None

_VALORES_DE_EJEMPLO = {"", "TU NOMBRE", "NOMBRE APELLIDO", "000000000", "123456789"}


class Alumno:
    """Datos de quien ejecuta el cuaderno. Se crea con identificar()."""

    def __init__(self, nombre, codigo, sesion):
        self.nombre = nombre
        self.codigo = codigo
        self.sesion = sesion

    @property
    def semilla(self):
        return int(self.codigo) % (2 ** 32)

    def rng(self, etiqueta=""):
        """Generador de números aleatorios propio del alumno.

        Cada etiqueta da una secuencia independiente: agregar un parámetro nuevo
        no cambia los que ya existían.
        """
        return np.random.default_rng([self.semilla, zlib.crc32(etiqueta.encode("utf-8"))])

    def __repr__(self):
        return f"Alumno({self.nombre!r}, {self.codigo!r}, {self.sesion!r})"


def identificar(nombre, codigo, sesion):
    """Valida la celda de identificación y devuelve el Alumno.

    Detiene el cuaderno con un mensaje claro si el nombre o el código faltan
    o se dejaron con el valor de ejemplo.
    """
    nombre = " ".join(str(nombre).split())
    codigo = str(codigo).strip()
    if nombre.upper() in _VALORES_DE_EJEMPLO or len(nombre.split()) < 2:
        raise ValueError("Escribe tu nombre completo (nombre y apellido) en NOMBRE y vuelve a ejecutar.")
    if codigo in _VALORES_DE_EJEMPLO or not codigo.isdigit() or not 7 <= len(codigo) <= 10:
        raise ValueError("Escribe tu código de alumno (sólo dígitos, de 7 a 10) en CODIGO y vuelve a ejecutar.")
    return Alumno(nombre, codigo, sesion)


def parametro(alumno, etiqueta, minimo, maximo, decimales=3):
    """Un valor personal entre minimo y maximo, siempre el mismo para el mismo código."""
    valor = alumno.rng(etiqueta).uniform(minimo, maximo)
    return round(float(valor), decimales)


def asignar(alumno, opciones, salto=0):
    """Elige una opción de la lista según el código (estación, ventana de fechas, carga del motor...)."""
    opciones = list(opciones)
    return opciones[(int(alumno.codigo) + salto) % len(opciones)]


def _redondea(valor):
    if isinstance(valor, (int, float, np.floating, np.integer)):
        return format(float(valor), ".4g")
    return str(valor)


def huella(alumno, resultados):
    """Seis caracteres que dependen del código, la sesión y los resultados (redondeados a 4 cifras)."""
    partes = [alumno.codigo, alumno.sesion] + [f"{k}={_redondea(v)}" for k, v in sorted(resultados.items())]
    return hashlib.sha256("|".join(partes).encode("utf-8")).hexdigest()[:6].upper()


def ahora():
    momento = _dt.datetime.now(_ZONA) if _ZONA else _dt.datetime.now()
    return momento.strftime("%Y-%m-%d %H:%M")


def titulo(ax, texto, alumno):
    """Título de la gráfica con el nombre del alumno."""
    ax.set_title(f"{texto} · {alumno.nombre}")


def firma(fig, alumno, resultados=None):
    """Pie de la figura: nombre · código · sesión · fecha y hora · huella.

    Toda gráfica que se entrega debe pasar por aquí.
    """
    texto = f"{alumno.nombre} · {alumno.codigo} · {alumno.sesion} · {ahora()}"
    if resultados:
        texto += f" · huella {huella(alumno, resultados)}"
    fig.text(0.01, 0.005, texto, fontsize=8, color="0.35", ha="left", va="bottom")
    fig.subplots_adjust(bottom=max(fig.subplotpars.bottom, 0.14))
    fig._firmada = True
    return fig


def exportar_csv(ruta, alumno, resultados, columnas, extra=None):
    """Escribe el CSV firmado que se importa en la pestaña «CSV» de la hoja de resultados.

    Renglones de metadatos: nombre, codigo, sesion, huella, ejecutado y los de `extra` (p. ej. la fuente
    de los datos). Después, un renglón por resultado («resultado:<nombre>», valor). Luego un renglón vacío
    y la tabla: encabezados y datos. `columnas` es un diccionario {encabezado: lista de valores}, todas de
    la misma longitud. La hoja busca cada dato por su etiqueta, no por su renglón.
    """
    claves = [k.lower() for k in resultados]
    if len(set(claves)) != len(claves):    # la hoja busca sin distinguir mayúsculas: U y u serían el mismo dato
        raise ValueError("Dos resultados tienen el mismo nombre salvo mayúsculas; renómbrelos.")
    largos = {len(v) for v in columnas.values()}
    if len(largos) != 1:
        raise ValueError("Todas las columnas de la tabla deben tener el mismo número de renglones.")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nombre", alumno.nombre])
        w.writerow(["codigo", alumno.codigo])
        w.writerow(["sesion", alumno.sesion])
        w.writerow(["huella", huella(alumno, resultados)])
        w.writerow(["ejecutado", ahora()])
        for k, v in (extra or {}).items():
            w.writerow([k, v])
        for k, v in resultados.items():
            w.writerow([f"resultado:{k}", _redondea(v)])
        w.writerow([])
        w.writerow(list(columnas))
        for fila in zip(*columnas.values()):
            w.writerow([_redondea(x) for x in fila])
    return ruta


def leer_csv_firmado(ruta):
    """Para el profesor: devuelve (metadatos, resultados, tabla) de un CSV firmado."""
    meta, resultados, tabla = {}, {}, []
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    i = 0
    while i < len(filas) and filas[i]:
        clave, valor = filas[i][0], filas[i][1]
        if clave.startswith("resultado:"):
            resultados[clave.split(":", 1)[1]] = float(valor)
        else:
            meta[clave] = valor
        i += 1
    tabla = filas[i + 1:] if i + 1 < len(filas) else []
    return meta, resultados, tabla


def verificar_csv(ruta):
    """Para el profesor: True si la huella del CSV corresponde a su código, sesión y resultados."""
    meta, resultados, _ = leer_csv_firmado(ruta)
    alumno = Alumno(meta["nombre"], meta["codigo"], meta["sesion"])
    return huella(alumno, resultados) == meta["huella"]


# ---------------------------------------------------------------------------------------------
# Revisión de lo que escribe el alumno. Comprueba que cada ejercicio esté COMPLETO, no que esté
# bien: el resultado correcto lo califica el profesor con la clave. Si falta algo, detiene el
# cuaderno y dice qué falta.
# ---------------------------------------------------------------------------------------------
import math
import re

REGISTRO = {}


def _normaliza(texto):
    t = str(texto).replace("\u202f", "").replace("\u00a0", "").replace(" ", "")
    t = re.sub(r"(?<![\d.])([1-9]\d{0,2})((?:,\d{3})+)(?![\d])",  # separador de miles: 32,000 -> 32000
               lambda m: m.group(1) + m.group(2).replace(",", ""), t)
    return t.replace(",", ".")                                       # coma decimal: 0,068 -> 0.068


def _formas(x):
    """Maneras razonables de escribir un número: entero, o redondeado a 2, 3 o 4 cifras significativas."""
    x = float(x)
    formas = set()
    if x == int(x) and abs(x) < 1e9:
        formas.add(str(int(x)))
    if abs(x) >= 100:
        formas.add(str(int(round(x))))                       # 34271.6 escrito como 34272
    if abs(x) >= 1:
        for dec in (1, 2):
            formas.add(f"{x:.{dec}f}".rstrip("0").rstrip("."))  # 123456.7 con sus decimales
    for cifras in (2, 3, 4, 5, 6):
        g = f"{x:.{cifras}g}"
        if "e" not in g:
            formas.add(g)
    return {f for f in formas if f not in ("", "-0")}


def aparece(x, texto):
    """True si el número x está escrito en el texto (sin confundir 0.03 con 0.0337)."""
    t = _normaliza(texto)
    def patron(f):
        cola = r"0*" if "." in f else r"(?:\.0*)?"            # 0.0342 = 0.034200 · 120 = 120.0, pero no 1200
        return r"(?<![\d.])" + re.escape(f) + cola + r"(?![\d])"
    return any(re.search(patron(f), t) for f in _formas(x))


def _palabras(texto):
    return len(re.findall(r"\w+", str(texto)))


def revisa(n, valor, sustitucion, prediccion, datos, unidades):
    """Revisión de un ejercicio de la parte A.

    valor: lo que calculó la expresión del alumno. sustitucion: texto con sus números, el resultado y la
    unidad (p. ej. «70 / 2**8 = 0.273 °C»). prediccion: qué espera ver en la práctica, con sus palabras.
    datos: los números personales que deben aparecer en la sustitución. unidades: unidades aceptadas.
    """
    faltan = []
    if valor is None:
        faltan.append("la expresión en Python")
    elif not isinstance(valor, (int, float, np.integer, np.floating)) or not math.isfinite(float(valor)):
        faltan.append("una expresión que dé un número")
    s = str(sustitucion)
    if "=" not in s:
        faltan.append("la sustitución con «=»")
    # datos: lista de números (se nombran por su valor) o {descripción: valor} (se nombran sin revelar el valor)
    pares = datos.items() if isinstance(datos, dict) else [(None, d) for d in datos]
    for que, d in pares:
        if not aparece(d, s):
            faltan.append(f"{que} en la sustitución" if que else f"tu dato {float(d):g} en la sustitución")
    if valor is not None and not faltan and not aparece(valor, s.split("=")[-1]):
        faltan.append("el resultado que dio tu expresión, después del último «=»")
    if not any(u in s for u in unidades):
        faltan.append("la unidad (" + " o ".join(unidades) + ")")
    if _palabras(prediccion) < 10:
        faltan.append("lo que predice, en al menos 10 palabras")
    REGISTRO[n] = not faltan
    if faltan:
        raise ValueError(f"Ejercicio {n}: falta " + "; ".join(faltan) + ".")
    print(f"✔ Ejercicio {n} completo.")


def revisa_texto(nombre, texto, numeros, minimo=20):
    """Revisión de una respuesta escrita: largo mínimo y los números propios que debe citar.

    numeros: diccionario {qué es: valor}; el mensaje nombra lo que falta sin revelar el valor.
    """
    faltan = [que for que, x in numeros.items() if not aparece(x, texto)]
    if _palabras(texto) < minimo:
        faltan.insert(0, f"al menos {minimo} palabras")
    REGISTRO[nombre] = not faltan
    if faltan:
        raise ValueError(f"{nombre}: falta " + "; ".join(faltan) + ".")
    print(f"✔ {nombre} completo.")


import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 3.6), 'axes.spines.top': False, 'axes.spines.right': False})

def completa(valor, que):
    if valor is None:
        raise ValueError(f'Completa {que} (✎) y vuelve a ejecutar.')
    return valor

R25, BETA, VCC = 10000.0, 3950.0, 5.0                 # termistor NTC 10 kΩ, β = 3950 K; alimentación de 5 V
N_BITS = 10                                            # convertidor de 10 bits
Q_V = VCC / 2 ** N_BITS                                # escalón: 5 V / 1024 = 4.883 mV (convención del curso)
N_MAX = 2 ** N_BITS - 1                                # cuenta máxima: 1023
T25_K = 298.15
BYTES_LINEA = 25                                       # «2027-02-05 08:00:00,0512» y el salto de línea
RF_OPCIONES = [2200, 3300, 4700, 6800, 10000, 15000, 22000, 33000, 47000, 68000]


def datos_s1(alumno):
    return dict(RF=asignar(alumno, RF_OPCIONES), T_OBJ=parametro(alumno, "T_obj", 15, 45, 1),
                DT_S=asignar(alumno, [10, 30, 60, 120, 300, 600], salto=1))


def r_ntc(t_c):
    return R25 * np.exp(BETA * (1 / (np.asarray(t_c) + 273.15) - 1 / T25_K))


def v_div(t_c, rf):
    r = r_ntc(t_c)
    return VCC * rf / (r + rf)


def t_de_cuenta(n, rf):
    """De la cuenta del ADC a la temperatura: la cadena al revés."""
    v = np.clip(np.asarray(n, float), 0.5, N_MAX - 0.5) * Q_V
    r = rf * (VCC / v - 1)
    return 1 / (1 / T25_K + np.log(r / R25) / BETA) - 273.15


def clave_s1(alumno):
    d = datos_s1(alumno)
    r = float(r_ntc(d["T_OBJ"]))
    v = VCC * d["RF"] / (r + d["RF"])
    return {"R_T": r, "V_T": v, "N_T": int(min(round(v / Q_V), N_MAX)), "kB_dia": 86400 / d["DT_S"] * BYTES_LINEA / 1000}

print('Herramientas del curso listas.')

## Identificación
Tu nombre y tu código aparecen en cada gráfica y en el CSV. Sin ellos el cuaderno no avanza.

In [ ]:
NOMBRE = ""   # ✎ tu nombre completo
CODIGO = ""   # ✎ tu código de alumno
SESION = "S1"

alumno = identificar(NOMBRE, CODIGO, SESION)
print("Cuaderno de", alumno.nombre, "· código", alumno.codigo)

## Parte A · Actividad en clase (30 min)
Datos comunes: termistor de `R25` = 10 000 Ω a 25 °C con `BETA` = 3950 K; alimentación `VCC` = 5 V; convertidor de
10 bits: el rango de 0 a `VCC` se divide en 2¹⁰ = 1024 escalones de `Q_V` = 5 V / 1024 = 4.883 mV, y la cuenta va
de 0 a `N_MAX` = 1023; cada renglón del archivo ocupa `BYTES_LINEA` = 25 bytes. Ejecuta la celda para
ver **tus** datos.

**Cómo se llena cada ejercicio.** Ejemplo con otros datos (no son los tuyos):
```
V_T    = 0.5556
SUST_2 = "5 × 1000 / (8000 + 1000) = 0.5556 V"
PRED_2 = "con una resistencia fija tan pequeña el voltaje de salida casi no cambiará con la temperatura"
```

In [ ]:
globals().update(datos_s1(alumno))
print(f"Tus datos · resistencia fija RF = {RF} Ω · temperatura de trabajo T_OBJ = {T_OBJ} °C · intervalo DT_S = {DT_S} s")

### ★ Ejercicio 1 · Del calor a la resistencia (3 pts)
**Para qué sirve:** es el primer eslabón: el sensor convierte la temperatura en una propiedad eléctrica.

La resistencia de un termistor NTC es **R = R25 · e^(β·(1/T − 1/T25))**, con las temperaturas en **kelvin**
(T = °C + 273.15; T25 = 298.15 K). En la calculadora, la exponencial es la tecla eˣ. Calcula `R_T` en Ω a tu `T_OBJ`.

In [ ]:
# ✎ 1a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
R_T = None   # ✎ escribe aquí
# ✎ 1b · La sustitución con TUS números, el resultado y la unidad
SUST_1 = ""   # ✎ escribe aquí
# ✎ 1c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_1 = ""   # ✎ escribe aquí

revisa(1, R_T, SUST_1, PRED_1, datos={'tu temperatura en kelvin': T_OBJ + 273.15}, unidades=['Ω', 'ohm'])

### Ejercicio 2 · De la resistencia al voltaje (2 pts)
**Para qué sirve:** el convertidor no lee ohms, lee volts; el acondicionamiento más sencillo es un divisor de tensión.

El termistor va arriba y `RF` abajo, hacia tierra. La salida es `VCC` por la **fracción de la resistencia total que
queda abajo**. Calcula `V_T` en V.

In [ ]:
# ✎ 2a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
V_T = None   # ✎ escribe aquí
# ✎ 2b · La sustitución con TUS números, el resultado y la unidad
SUST_2 = ""   # ✎ escribe aquí
# ✎ 2c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_2 = ""   # ✎ escribe aquí

revisa(2, V_T, SUST_2, PRED_2, datos={'tu dato RF': RF, 'tu R_T': R_T}, unidades=['V'])

### Ejercicio 3 · Del voltaje a la cuenta (2 pts)
**Para qué sirve:** lo que se guarda no es un voltaje sino un número entero: la cuenta del convertidor.

El convertidor divide el rango de 0 a `VCC` en 2¹⁰ = 1024 escalones de `Q_V` = `VCC` / 1024. La cuenta es tu
voltaje dividido entre el escalón, **redondeada** a entero (sin decimales); la más grande posible es `N_MAX` = 1023.

In [ ]:
# ✎ 3a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
N_T = None   # ✎ escribe aquí
# ✎ 3b · La sustitución con TUS números, el resultado y la unidad
SUST_3 = ""   # ✎ escribe aquí
# ✎ 3c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_3 = ""   # ✎ escribe aquí

revisa(3, N_T, SUST_3, PRED_3, datos={'tu V_T': V_T}, unidades=['cuenta'])

### Ejercicio 4 · ¿Cuántos datos genera un sensor al día? (2 pts)
**Para qué sirve:** en la industria se registran cientos de variables; el volumen decide el almacenamiento y la
forma de analizar.

Calcula cuántos renglones se escriben en un día (86 400 s) con tu intervalo y cuántos **kB** ocupan (1 kB = 1000 bytes).

In [ ]:
# ✎ 4a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
kB_dia = None   # ✎ escribe aquí
# ✎ 4b · La sustitución con TUS números, el resultado y la unidad
SUST_4 = ""   # ✎ escribe aquí
# ✎ 4c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_4 = ""   # ✎ escribe aquí

revisa(4, kB_dia, SUST_4, PRED_4, datos={'tu dato DT_S': DT_S}, unidades=['kB'])

In [ ]:
T = np.linspace(0, 50, 201)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(T, r_ntc(T) / 1000); a1.plot(T_OBJ, R_T / 1000, "o", color="#D9822B")
a1.set_xlabel("T (°C)"); a1.set_ylabel("R del termistor (kΩ)"); titulo(a1, "Sensor", alumno)
a2.plot(T, v_div(T, RF)); a2.plot(T_OBJ, V_T, "o", color="#D9822B")
a2.set_xlabel("T (°C)"); a2.set_ylabel("salida del divisor (V)"); a2.set_title(f"Divisor con RF = {RF} Ω")
firma(fig, alumno); plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

### Cierre (1 pt)
Describe tu cadena completa con tus números: a tu temperatura, ¿qué resistencia, qué voltaje y qué cuenta llegan al
archivo? Al menos 20 palabras, con tu voltaje y tu cuenta.

In [ ]:
CIERRE = ""   # ✎ escribe aquí

revisa_texto('Cierre', CIERRE, {'tu voltaje': V_T, 'tu cuenta': N_T})

## Parte B · Práctica (80 min)
**Objetivo:** pasar datos reales por tu cadena y medir cuánta información se pierde al guardar cuentas enteras.

Una semana de temperatura de Guadalajara, hora por hora ([Open-Meteo](https://open-meteo.com/), licencia CC BY 4.0);
la semana depende de tu código.

| Bloque | min | Qué haces |
|---|---|---|
| B1 · Datos | 10 | Descargas la semana |
| B2 · Por tu cadena | 20 | Cada hora pasa a resistencia, voltaje y cuenta, como lo guardaría el registrador |
| B3 · De regreso | 20 | Calculas cuánto vale una cuenta en volts; el cuaderno reconstruye la temperatura; predices y mides el error |
| B4 · Conclusión | 20 | Comparas predicho contra obtenido, con tus números |
| Tabla y captura | 10 | Descargas el CSV y lo importas en tu hoja de resultados |

In [ ]:
import json, urllib.request, datetime as dt
lunes = dt.date(2025, 1, 6) + dt.timedelta(weeks=int(alumno.codigo) % 50)
domingo = lunes + dt.timedelta(days=6)
url = ("https://archive-api.open-meteo.com/v1/archive?latitude=20.67&longitude=-103.35"
       f"&start_date={lunes}&end_date={domingo}&hourly=temperature_2m&timezone=America%2FMexico_City")
try:
    with urllib.request.urlopen(url, timeout=30) as resp:
        d = json.load(resp)["hourly"]
    fechas, T_real = d["time"], np.array(d["temperature_2m"], dtype=float)
    FUENTE = "Open-Meteo (ERA5), Guadalajara"
except Exception as e:
    print("⚠ No se pudo descargar de Open-Meteo:", type(e).__name__, e)
    print("⚠ Se usan datos SINTÉTICOS. Vuelve a ejecutar esta celda con internet antes de entregar.")
    hh = np.arange(7 * 24)
    fechas = [f"hora {i}" for i in hh]
    T_real = 20 + 7 * np.sin(2 * np.pi * (hh - 9) / 24) + alumno.rng("ruido").normal(0, 0.6, hh.size)
    FUENTE = "SINTÉTICOS (sin conexión)"
print(f"B1 · Semana del {lunes} al {domingo} · {len(T_real)} lecturas · fuente: {FUENTE}")

In [ ]:
# B2 · Por tu cadena: temperatura → resistencia → voltaje → cuenta entera
cuentas = np.minimum(np.round(v_div(T_real, RF) / Q_V), N_MAX).astype(int)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
a1.plot(T_real); a1.set_ylabel("T real (°C)"); titulo(a1, "Lo que pasó", alumno)
a2.step(range(len(cuentas)), cuentas, where="mid", color="#D9822B"); a2.set_ylabel("cuenta del ADC")
a2.set_xlabel("hora"); a2.set_title("Lo que guarda el registrador")
firma(fig, alumno); plt.show()
print(f"Cuentas de {cuentas.min()} a {cuentas.max()}: la semana usó {cuentas.max() - cuentas.min() + 1} de las 1024 posibles")

### B3 · De regreso: de la cuenta a la temperatura
1. `PASO_V`: cuántos volts vale **una** cuenta: el escalón, el rango de 0 a `VCC` repartido en 2¹⁰ = 1024. El cuaderno
   multiplica cada cuenta por tu `PASO_V`, y con eso calcula la resistencia y la temperatura reconstruida.
2. `resol_pred_C`: el **error máximo predicho**: medio escalón expresado en °C. Divide tu `PASO_V` entre la
   sensibilidad `S_V_C` (impresa arriba, en V/°C) y toma la mitad.

In [ ]:
T_media = float(np.mean(T_real))
S_V_C = float((v_div(T_media + 0.5, RF) - v_div(T_media - 0.5, RF)))   # V por °C a la temperatura media
print(f"Temperatura media de tu semana: {T_media:.2f} °C · sensibilidad del divisor S_V_C = {S_V_C:.6f} V/°C ({S_V_C * 1000:.3f} mV/°C)")

In [ ]:
PASO_V = None   # ✎ escribe aquí
resol_pred_C = None   # ✎ escribe aquí

completa(PASO_V, "cuántos volts vale una cuenta"); completa(resol_pred_C, "el error máximo predicho")
V_B = cuentas * PASO_V
R_B = RF * (VCC / np.asarray(V_B) - 1)
T_rec = 1 / (1 / T25_K + np.log(R_B / R25) / BETA) - 273.15
error = T_rec - T_real
error_obt_C = float(np.max(np.abs(error)))
RESULTADOS = {"R_T": float(R_T), "V_T": float(V_T), "N_T": int(N_T), "kB_dia": float(kB_dia),
              "resol_pred_C": float(resol_pred_C), "error_obt_C": error_obt_C}
fig, ax = plt.subplots()
ax.plot(error, lw=1, label="error de la temperatura reconstruida")
ax.axhline(resol_pred_C, ls="--", c="r", label="± error predicho"); ax.axhline(-resol_pred_C, ls="--", c="r")
ax.set_xlabel("hora"); ax.set_ylabel("error (°C)"); ax.legend(fontsize=8, loc="upper right")
titulo(ax, "Lo que se pierde al guardar cuentas", alumno)
firma(fig, alumno, RESULTADOS); plt.show()
print(f"Error máximo predicho {resol_pred_C:.4f} °C · obtenido {error_obt_C:.4f} °C")

### B4 · Conclusión
¿El error obtenido quedó cerca del predicho? ¿Por qué no es exactamente igual en toda la semana? Al menos 25
palabras, con tus dos valores.

In [ ]:
CONCLUSION_B = ""   # ✎ escribe aquí

revisa_texto('Conclusión de la práctica', CONCLUSION_B,
             {'el error predicho': resol_pred_C, 'el error obtenido': error_obt_C}, minimo=25)

### Tabla y captura · CSV para tu hoja de resultados
La celda descarga `S1_<código>.csv`. En tu hoja de resultados: abre la pestaña **CSV** → **Archivo → Importar →
Subir** → elige el archivo → **Reemplazar la hoja actual**. La pestaña Resultados se llena sola.

In [ ]:
import os
ruta = f"S1_{alumno.codigo}.csv"
exportar_csv(ruta, alumno, RESULTADOS,
             {"hora": np.arange(len(T_real)), "T_real_C": T_real, "cuenta": cuentas, "T_reconstruida_C": T_rec},
             extra={"fuente": FUENTE, "semana": f"{lunes} a {domingo}", "rf_ohm": RF})
print("Archivo listo:", ruta, "· huella", huella(alumno, RESULTADOS), f"· {os.path.getsize(ruta) / 1000:.1f} kB")
try:
    from google.colab import files
    files.download(ruta)
except ImportError:
    pass

## Cierre del cuaderno
Antes de entregar: **Entorno de ejecución → Reiniciar y ejecutar todo**. Todo debe salir ✔.

In [ ]:
for k, ok in REGISTRO.items():
    print(("✔" if ok else "✘"), k if isinstance(k, str) else f"Ejercicio {k}")
print(f"\n{alumno.nombre} · {alumno.codigo} · {SESION} · {ahora()} · fuente: {FUENTE} · huella {huella(alumno, RESULTADOS)}")